In [1]:
import torch
import pandas as pd
import pickle
from tqdm import tqdm
from transformers import AutoTokenizer, EsmModel


In [12]:
csv_path = '/home/lherrmann/projects/enzyme-design/flip2_Project/test/csv_id/pickel/test.csv'
output_path = "/home/lherrmann/projects/enzyme-design/flip2_Project/test/csv_id/pickel/esm2_embeddings.pkl"
model_name = "facebook/esm2_t33_650M_UR50D" # Standard baseline (1280 dims)
device = torch.device("cuda" if torch.cuda.is_available() else print('error'))


In [6]:
df = pd.read_csv(csv_path)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmModel.from_pretrained(model_name).to(device)
model.eval()

Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(33, 1280, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-32): 33 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=1280, out_features=1280, bias=True)
            (key): Linear(in_features=1280, out_features=1280, bias=True)
            (value): Linear(in_features=1280, out_features=1280, bias=True)
            (rotary_embeddings): RotaryEmbedding()
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=1280, out_features=1280, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        )
        (intermediate): EsmIntermediate(
          (dense): Linear(in_features=1280, out_features=5120, bias=True)
        )
        (output): EsmOut

In [13]:
# 3. Generate Embeddings
embeddings_dict = {}

print(f"Generating embeddings using {device}...")
with torch.no_grad():
    for _, row in tqdm(df.iterrows(), total=len(df)):
        prot_id = row['id']
        sequence = row['sequence']
        
        # Tokenize and move to GPU
        inputs = tokenizer(sequence, return_tensors="pt", truncation=True, max_length=1024).to(device)
        
        # Forward pass
        outputs = model(**inputs)
        
        # Get 'mean' embedding (average over the sequence length)
        # Resulting vector size: 1280 for the t33 model
        mean_embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
        
        embeddings_dict[prot_id] = mean_embedding

# 4. Save to temporary folder/file
with open(output_path, "wb") as f:
    pickle.dump(embeddings_dict, f)

print(f"Success! Saved {len(embeddings_dict)} embeddings to {output_path}")

Generating embeddings using cuda...


100%|██████████| 49/49 [00:06<00:00,  7.91it/s]

Success! Saved 49 embeddings to /home/lherrmann/projects/enzyme-design/flip2_Project/test/csv_id/pickel/esm2_embeddings.pkl
